# Path C+ Smoke Test — V5-mini avec 17 P0 fixes appliqués

**Objectif** : valider que les 17 fixes Path C+ cassent le pattern band-diagonal
Q_phys=0.40 observé en V5-mini avant correction.

**Cible** : ≥ 10 epochs, ≤ 1 heure sur Colab T4 (smoke test rapide, pas un full
retrain).

**Critères de succès** (AI eng round 7) :
- ✅ `Q_phys > 0.55` par epoch 10 (vs 0.40 collapsed)
- ✅ `skip_block.parameters().grad.norm() > 0` après epoch 1 (J8 fix)
- ✅ AUCUN warning `J8 skip_block forward failed` (contract broken si présent)
- ✅ AUCUN warning `§1.6 missing-edge fallback` (cfg à zéro gradient si présent)
- ⚠️ Si `Q_phys < 0.45` à epoch 10 → bisecter entre commits J3, I11, J8

**Branch** : `four-node-causal` (23 commits depuis `two-stage-causal`)

## Phase plan
1. Bootstrap Colab (clone, install deps)
2. GPU profile detection
3. Pre-flight check (K8 temporal split warning)
4. Build V5-mini stack
5. Run smoke test 10 epochs avec instrumentation
6. Verdict + go/no-go pour Batch D


In [ ]:
# >>> COLAB_BOOTSTRAP
# Bootstrap Colab optimisé — premier run ~3 min, re-runs ~30 s.
# Stratégie :
#   • Code sur SSD local (/content/) — git clone 5-10× plus rapide que vers Drive.
#   • Drive UNIQUEMENT pour les checkpoints (cf. cellule helpers plus bas).
#   • Pas de ``pip install -r requirements.txt`` brut (déclenche la compilation
#     CUDA de torch-scatter/torch-sparse → 20-30 min). À la place : install
#     pinned des seules deps non pré-installées par Colab.
#   • Wheels PyG pré-construits via le bon index (sinon torch-scatter/torch-sparse
#     compilent depuis les sources — interdit ici).
#   • GIT SYNC: pull obligatoire (fetch + reset hard vers origin/<branche>)
#     pour garantir HEAD = remote HEAD (Batch D + E + F + F-bis inclus).
#
# Hors Colab : no-op.
import os, sys, subprocess, time, shlex
from pathlib import Path

# ── Configuration utilisateur ──────────────────────────────────────────
GIT_URL: str | None = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH: str = "four-node-causal"  # Path C+ branch (Batches A-F + blindspots)
LOCAL_PROJECT = "/content/climate_data"  # SSD — toujours rapide
GIT_PULL_ON_RESUME = True
SKIP_PIP_IF_IMPORTABLE = True  # si ``import st_cdgm`` réussit déjà → skip pip

_IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()


def _run(cmd: str, *, check: bool = True, timeout: int | None = None) -> int:
    """Exécute une commande shell avec timing visible."""
    print(f"$ {cmd}")
    t0 = time.time()
    rc = subprocess.call(shlex.split(cmd), timeout=timeout)
    dt = time.time() - t0
    print(f"  ↳ rc={rc}  ({dt:.1f}s)")
    if check and rc != 0:
        raise RuntimeError(f"Commande échouée : {cmd!r} (rc={rc})")
    return rc


if _IS_COLAB:
    _T0 = time.time()
    print("🛰️  Colab détecté — bootstrap en cours…\n")

    # 1) Monter Drive (idempotent — pour la cellule de persistance plus loin)
    from google.colab import drive  # type: ignore[import-not-found]
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    else:
        print("   /content/drive déjà monté.")

    # 2) Clone vers SSD (PAS vers Drive — FUSE est lent pour des milliers de petits fichiers)
    project_path = Path(LOCAL_PROJECT)
    if not (project_path / ".git").exists():
        if GIT_URL is None:
            raise RuntimeError(
                "GIT_URL=None et projet absent du SSD. Renseignez GIT_URL ci-dessus, "
                "ou pré-uploadez le projet à " + LOCAL_PROJECT + " avant ce run."
            )
        project_path.parent.mkdir(parents=True, exist_ok=True)
        # NB: --depth 1 garde le clone rapide ; le pull/reset ci-dessous garantit
        # ensuite que HEAD = origin/<branche> HEAD (Batch D + E + F + F-bis inclus).
        _run(f"git clone --depth 1 -b {GIT_BRANCH} {GIT_URL} {LOCAL_PROJECT}")

    if GIT_PULL_ON_RESUME:
        # === Garantir que /content/climate_data = origin/<branche> HEAD ===
        # Aucun travail local n'est attendu sur le SSD Colab (le code authoritative
        # vit sur GitHub), donc on FORCE un reset hard vers origin/<branche>
        # plutôt que `pull --ff-only` qui échoue silencieusement en cas de
        # divergence locale ou d'état détaché.
        print()
        print("=" * 70)
        print(f"GIT SYNC — force {LOCAL_PROJECT} to origin/{GIT_BRANCH}")
        print("=" * 70)

        # Branche actuelle (peut être autre chose si un précédent run l'a switchée)
        try:
            current_branch = subprocess.check_output(
                shlex.split(f"git -C {LOCAL_PROJECT} rev-parse --abbrev-ref HEAD")
            ).decode().strip()
        except Exception:
            current_branch = "(unknown)"
        print(f"   Current branch : {current_branch}")

        # Fetch obligatoire — blindspot #8: depth=200 (was 50) pour absorber
        # plusieurs Batches simultanés sans risquer d'historique tronqué.
        _run(f"git -C {LOCAL_PROJECT} fetch --depth=200 origin {GIT_BRANCH}",
             timeout=180, check=True)

        # Checkout si la branche locale n'est pas la bonne ; sinon reset hard.
        if current_branch != GIT_BRANCH:
            print(f"   Switching from {current_branch} to {GIT_BRANCH}...")
            _run(f"git -C {LOCAL_PROJECT} checkout -B {GIT_BRANCH} origin/{GIT_BRANCH}",
                 timeout=30, check=True)
        else:
            # Reset hard vers origin/<branche> -> garantit HEAD = remote HEAD
            # même si pull --ff-only échouerait (état détaché, divergence, etc.)
            _run(f"git -C {LOCAL_PROJECT} reset --hard origin/{GIT_BRANCH}",
                 timeout=30, check=True)

        # Echo HEAD pour vérification visuelle. Pour smoke #4 on attend
        # au minimum 5a4b6ca (Batch F-bis) ou plus récent.
        try:
            head_sha = subprocess.check_output(
                shlex.split(f"git -C {LOCAL_PROJECT} rev-parse --short HEAD")
            ).decode().strip()
            head_msg = subprocess.check_output(
                shlex.split(f"git -C {LOCAL_PROJECT} log -1 --pretty=%s")
            ).decode().strip()
            print(f"   HEAD = {head_sha}  ({head_msg})")
            print(f"   Last 12 commits on {GIT_BRANCH}:")
            log_out = subprocess.check_output(
                shlex.split(f"git -C {LOCAL_PROJECT} log -12 --oneline")
            ).decode().strip()
            for _line in log_out.splitlines():
                print(f"     {_line}")
            # Hash list of all council-blocking commits (Batches D, E, F, F-bis).
            EXPECTED_HASHES = [
                "5a4b6ca",  # Batch F-bis (council follow-up nits)
                "876a082",  # Batch F (gate + Q_phys variants + projection hook + PC5-PC8)
                "b2cc50e",  # Batch E (council follow-up Batch D)
                "d47fe2f",  # K5
                "46c5791",  # K9
                "f5159df",  # K3
                "7e9630c",  # K2
                "7d81079",  # J29
            ]
            present = [h for h in EXPECTED_HASHES if h in log_out]
            print(f"   Expected D+E+F+F-bis commits present: {len(present)}/{len(EXPECTED_HASHES)}")
            print(f"     {present}")
            if len(present) < 7:
                print(f"   ⚠️  WARNING: only {len(present)}/{len(EXPECTED_HASHES)} expected commits found.")
                print(f"       Verify origin/{GIT_BRANCH} has been pushed (5a4b6ca or newer).")
        except Exception as e:
            print(f"   (HEAD verification failed: {e})")
        print()

    # 3) cd dans la racine — config/, src/, etc. en chemins relatifs
    os.chdir(project_path)
    print(f"   chdir → {os.getcwd()}\n")

    # 4) Test d'import — si st_cdgm marche déjà, on saute pip (énorme gain au re-run)
    src_path = str(project_path / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    _need_pip = True
    if SKIP_PIP_IF_IMPORTABLE:
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            from diffusers import UNet2DConditionModel  # noqa: F401
            import torch_geometric  # noqa: F401
            _need_pip = False
            print("✓ Imports critiques OK — pip install sauté.")
        except ImportError as _imp_err:
            print(f"   import st_cdgm a échoué ({_imp_err}) — pip install requis.")

    if _need_pip:
        # 5) Versions PyTorch / CUDA déjà installées par Colab
        import torch
        TORCH_VER = torch.__version__.split("+")[0]  # ex. "2.5.1"
        TORCH_TAG = f"torch-{TORCH_VER}"             # ex. "torch-2.5.1"
        CUDA_TAG = "cu" + (torch.version.cuda or "121").replace(".", "") if torch.cuda.is_available() else "cpu"
        print(f"   torch={TORCH_VER}, cuda={CUDA_TAG}\n")

        # 6) Install des deps NON pré-installées par Colab
        EXTRA_DEPS = [
            "omegaconf==2.3.0",
            "hydra-core==1.3.2",
            "diffusers==0.36.0",
            "transformers==4.57.6",
            "accelerate==1.12.0",
            "huggingface-hub==0.36.0",
            "safetensors==0.7.0",
            "xbatcher",
            "webdataset",
            "cftime",
            "h5netcdf",
            "numcodecs",
            "torch-geometric",
            "xformers",
        ]
        deps_str = " ".join(shlex.quote(p) for p in EXTRA_DEPS)
        _run(
            f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location {deps_str}",
            timeout=600,
        )

        # 8) Editable install du package
        _run(
            f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
            f"--no-deps -e {LOCAL_PROJECT}",
            timeout=120,
        )

        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            print("✓ st_cdgm importable.")
        except ImportError as e:
            print(f"⚠️  st_cdgm pas encore importable depuis ce kernel : {e}")
            print("   → Probablement un cache d'import — Runtime → Restart runtime puis re-run.")

    print(f"\n✅ Bootstrap Colab terminé en {time.time() - _T0:.1f}s.")

else:
    _here = Path.cwd()
    for _candidate in [_here, *_here.parents]:
        if (_candidate / "config" / "training_config.yaml").exists() and (_candidate / "setup.py").exists():
            if _candidate != _here:
                os.chdir(_candidate)
                print(f"📂 chdir → {os.getcwd()} (racine projet détectée)")
            break
    print("ℹ️  Hors Colab — bootstrap sauté (assume install déjà faite).")

# === Path C+ specific smoke test (apres bootstrap) ===
print()
print("=" * 70)
print("Path C+ Verifications")
print("=" * 70)

# Verify Path C+ fixes are present
try:
    from st_cdgm.models import ConditionalSkipBlock
    print("[Path C+] ConditionalSkipBlock importable (J8 fix target)")
except ImportError as e:
    print(f"[Path C+] ConditionalSkipBlock NON disponible : {e}")

try:
    from st_cdgm.models.intelligible_encoder import IntelligibleVariableEncoder
    import inspect
    src = inspect.getsource(IntelligibleVariableEncoder)
    if "metapath_convs" in src and "layer_norms" in src:
        print("[Path C+] §1.6 + J3 fixes detected (metapath_convs + per-metapath LayerNorm)")
    else:
        print("[Path C+] WARN : §1.6 + J3 fixes NOT detected in encoder source")
except Exception as e:
    print(f"[Path C+] encoder check failed : {e}")

try:
    from st_cdgm.models.causal_rcn import CASTLEAnchor
    import inspect
    src = inspect.getsource(CASTLEAnchor)
    if "H_t_detached" in src:
        print("[Path C+] I11 fix detected (CASTLE H_t.detach)")
    else:
        print("[Path C+] WARN : I11 fix NOT detected in CASTLE source")
except Exception as e:
    print(f"[Path C+] CASTLE check failed : {e}")

# Batch D + F verifications
try:
    import inspect as _ins
    from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder as _CDD
    if "J29 audit fix" in _ins.getsource(_CDD):
        print("[Path C+] J29 fix detected (cfg_scale assertion in edm_karras path)")
    from st_cdgm.data.pipeline import NetCDFDataPipeline as _NDP
    _src_pipeline = _ins.getsource(_NDP)
    if "K9 fix" in _src_pipeline and "train_start_date" in _src_pipeline:
        print("[Path C+] K9 fix detected (temporal split via xarray.sel)")
    if "K5 fix" in _src_pipeline:
        print("[Path C+] K5 fix detected (train-only normalisation stats)")
    from st_cdgm.evaluation.evaluation_xai import evaluate_metrics as _em
    if "f1_climatology" in _ins.signature(_em).parameters:
        print("[Path C+] K2 fix detected (f1_climatology plumbing through evaluate_metrics)")
    # Batch F-1 verification (DEFAULT_HYPERPARAMS gate auto-scale)
    from scripts.finetune_stage1_bundle_b import DEFAULT_HYPERPARAMS as _DH
    if _DH.get("dag_gate_warmup_start_epoch") is None:
        print("[Path C+] Batch F-1 detected (DEFAULT_HYPERPARAMS gate=None -> auto-scale)")
    else:
        print(f"[Path C+] WARN : Batch F-1 NOT applied (gate start={_DH.get('dag_gate_warmup_start_epoch')})")
except Exception as e:
    print(f"[Path C+] Batch D/F verification failed : {e}")

try:
    from path_c_plus.scripts.gpu_detect import detect_gpu_profile
    from path_c_plus.scripts.preflight_checks import run_all_preflight_checks
    print("[Path C+] gpu_detect + preflight_checks helpers OK")
except ImportError as e:
    print(f"[Path C+] path_c_plus helpers NON disponibles : {e}")

print()


In [ ]:
# === Cell 2 : GPU profile detection + pre-flight K8 check ===
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner
from path_c_plus.scripts.preflight_checks import run_all_preflight_checks
from omegaconf import OmegaConf

GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)

# Load config + run pre-flight checks
CONFIG = OmegaConf.load("config/training_config.yaml")
_corrdiff = OmegaConf.load("config/training_config_corrdiff_normal.yaml")
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

print()
print("=" * 70)
print("Pre-flight checks (Path C+ Phase 0.3)")
print("=" * 70)
preflight_report = run_all_preflight_checks(CONFIG)
print()
print(f"Pre-flight report: {preflight_report}")
print()
print("NOTE: K8 temporal split warning is EXPECTED (commit 9402cfc declared")
print("      fields, K9/K5 enforcement deferred to Batch D commits 24-25).")
print("      Smoke test runs on FULL dataset for now.")


In [ ]:
# === Cell 3 : Build V5-mini stack + load baseline checkpoint ===
import torch
import numpy as np
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Paths
V5_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_v2_corrdiff_normal")
SMOKE_DIR = Path("/content/drive/MyDrive/climate_data/smoke_test_v6_pathc")
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

# Copy baseline checkpoint as smoke starting point
import shutil as _sh
_smoke_ckpt = SMOKE_DIR / "epoch_last.pth"
if not _smoke_ckpt.exists():
    _src = V5_DIR / "epoch_last.pth"
    print(f"[Setup] Copying baseline {_src.name} to smoke dir...")
    _sh.copy(_src, _smoke_ckpt)
    print(f"  [OK] {_smoke_ckpt.stat().st_size/1024**3:.2f} GB")

# Build the stack via the same code path as the eval notebook
# (we vendor a minimal version here to keep smoke test self-contained)
print()
print("=" * 70)
print("Building V5-mini stack with NEW Path C+ code")
print("=" * 70)

# Apply GPU profile to config
CONFIG.training.batch_size = GPU_PROFILE["batch_size"]
CONFIG.training.use_amp = GPU_PROFILE["use_amp"]
CONFIG.training.num_workers = GPU_PROFILE["num_workers"]

# Build pipeline + builder (in-distribution ACCESS-CM2)
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models import (
    IntelligibleVariableEncoder, IntelligibleVariableConfig,
    GraphToGridDecoder, RCNCell, RCNSequenceRunner,
    CausalDiffusionDecoder,
)
from st_cdgm.models.edm_preconditioner import EDMConfig
try:
    from st_cdgm.models import ConditionalSkipBlock
    SKIP_AVAILABLE = True
except ImportError:
    SKIP_AVAILABLE = False
    ConditionalSkipBlock = None

# Data root detection (Drive or local SSD)
DATA_ROOT = Path("/content/drive/MyDrive/climate_data/data")
LR_PATH = str(DATA_ROOT / "train/predictor_ACCESS-CM2_hist.nc")
HR_PATH = str(DATA_ROOT / "train/pr_ACCESS-CM2_hist.nc")
STATIC_PATH = str(DATA_ROOT / "static_predictors/ERA5_eval_ccam_12km.198110_NZ_Invariant.nc")
MEAN_PATH = str(DATA_ROOT / "normalization_coefs/mean_1974_2011.nc")
STD_PATH = str(DATA_ROOT / "normalization_coefs/std_1974_2011.nc")

pipeline = NetCDFDataPipeline(
    lr_path=LR_PATH, hr_path=HR_PATH, static_path=STATIC_PATH,
    seq_len=int(CONFIG.data.seq_len),
    baseline_strategy=str(CONFIG.data.baseline_strategy),
    baseline_factor=int(CONFIG.data.baseline_factor),
    target_transform=str(CONFIG.data.get("target_transform", "log1p")),
    normalize=bool(CONFIG.data.normalize),
    nan_fill_strategy=str(CONFIG.data.nan_fill_strategy),
    precipitation_delta=float(CONFIG.data.get("precipitation_delta", 0.01)),
    lr_variables=list(CONFIG.data.lr_variables),
    hr_variables=list(CONFIG.data.hr_variables),
    static_variables=list(CONFIG.data.static_variables),
    means_path=MEAN_PATH, stds_path=STD_PATH,
    eager_load_datasets=False,
)

lr_shape = tuple(CONFIG.graph.lr_shape)
hr_shape = tuple(CONFIG.graph.hr_shape)
builder = HeteroGraphBuilder(
    lr_shape=lr_shape, hr_shape=hr_shape,
    static_dataset=pipeline.get_static_dataset(),
    include_mid_layer=bool(CONFIG.graph.include_mid_layer),
)
print(f"[OK] Builder created: {len(builder.dynamic_node_types)} dyn + {len(builder.static_node_types)} static nodes")
print(f"     LR shape: {lr_shape}, HR shape: {hr_shape}")


In [ ]:
# === Cell 4 : Materialize datasets + load stack from baseline ===
import itertools as _it
from torch.utils.data import Dataset as _TorchDataset


class _MapStyleListDataset(_TorchDataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        return self.samples[i]


# Smoke test: 100 train + 24 val samples (sufficient for 10 epochs at bs=8)
MAX_TRAIN_SAMPLES = 100
MAX_VAL_SAMPLES = 24

print(f"[Smoke] Materializing {MAX_TRAIN_SAMPLES} train + {MAX_VAL_SAMPLES} val samples...")
train_iter = pipeline.build_sequence_dataset(
    seq_len=int(CONFIG.data.seq_len), stride=1, as_torch=True,
)
_train_samples = list(_it.islice(train_iter, MAX_TRAIN_SAMPLES))
train_dataset = _MapStyleListDataset(_train_samples)
print(f"  [OK] train_dataset : {len(train_dataset)} samples")

val_iter = pipeline.build_sequence_dataset(
    seq_len=int(CONFIG.data.seq_len),
    stride=int(CONFIG.data.stride),
    as_torch=True,
)
_val_samples = list(_it.islice(val_iter, MAX_VAL_SAMPLES))
val_dataset = _MapStyleListDataset(_val_samples)
print(f"  [OK] val_dataset   : {len(val_dataset)} samples")

# Detect runtime driver_dim from samples
_runtime_dim = int(_train_samples[0]["lr"].shape[1])
if _runtime_dim != int(CONFIG.rcn.driver_dim):
    CONFIG.rcn.driver_dim = _runtime_dim
    CONFIG.rcn.reconstruction_dim = _runtime_dim


def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    dynamic_features = {nt: lr_nodes_steps[0] for nt in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    return {"lr": lr_tensor, "residual": sample["residual"],
            "baseline": sample.get("baseline"), "hetero": hetero}


# Build encoder + RCN + regression head + skip_block + diffusion (V5-mini config)
allowed_nodes = set(builder.dynamic_node_types + builder.static_node_types)
encoder_configs = []
for _mp in CONFIG.encoder.metapaths:
    _src, _rel, _tgt = _mp.src, _mp.relation, _mp.target
    if _src in allowed_nodes and _tgt in allowed_nodes:
        encoder_configs.append(IntelligibleVariableConfig(
            name=_mp.name,
            meta_path=(_src, _rel, _tgt),
            pool=_mp.get("pool", "mean"),
        ))
if pipeline.get_static_dataset() is not None:
    encoder_configs.append(IntelligibleVariableConfig(
        name="static", meta_path=("SP_HR", "causes", "GP850"), pool="mean",
    ))

print(f"[Stack] {len(encoder_configs)} encoder configs (= q variables)")

enc = IntelligibleVariableEncoder(
    configs=encoder_configs,
    hidden_dim=CONFIG.encoder.hidden_dim,
    conditioning_dim=CONFIG.encoder.conditioning_dim,
).to(DEVICE)
num_vars = len(encoder_configs)
print(f"  [§1.6 + J4] metapath_convs: {len(enc.metapath_convs)} distinct SAGEConvs")
print(f"  [J3]        layer_norms:   {len(enc.layer_norms)} distinct LayerNorms")

rcn_cell = RCNCell(
    num_vars=num_vars,
    hidden_dim=CONFIG.rcn.hidden_dim,
    driver_dim=int(CONFIG.rcn.driver_dim),
    reconstruction_dim=int(CONFIG.rcn.reconstruction_dim),
    dropout=CONFIG.rcn.dropout,
).to(DEVICE)
rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))

rh = GraphToGridDecoder(
    d_model=CONFIG.encoder.hidden_dim,
    hr_h=CONFIG.graph.hr_shape[0], hr_w=CONFIG.graph.hr_shape[1],
).to(DEVICE)

edm_cfg = EDMConfig.from_yaml_dict(CONFIG.diffusion.get("edm", {}))
_unet_kwargs = OmegaConf.to_container(CONFIG.diffusion.unet_kwargs, resolve=True)
for _k in ("down_block_types", "up_block_types"):
    if _k in _unet_kwargs and isinstance(_unet_kwargs[_k], list):
        _unet_kwargs[_k] = tuple(_unet_kwargs[_k])
hr_channels = int(_train_samples[0]["residual"].shape[1])
diff = CausalDiffusionDecoder(
    in_channels=hr_channels,
    conditioning_dim=CONFIG.diffusion.conditioning_dim,
    height=int(CONFIG.diffusion.height),
    width=int(CONFIG.diffusion.width),
    unet_kwargs=_unet_kwargs,
    scheduler_type=str(CONFIG.diffusion.scheduler_type),
    use_gradient_checkpointing=bool(CONFIG.diffusion.get("use_gradient_checkpointing", False)),
    conv_padding_mode=str(CONFIG.diffusion.get("conv_padding_mode", "zeros")),
    anti_checkerboard=bool(CONFIG.diffusion.get("anti_checkerboard", False)),
    edm_config=edm_cfg,
    causal_concat=True,
).to(DEVICE)

skip = None
if SKIP_AVAILABLE:
    skip = ConditionalSkipBlock(
        lr_channels=len(CONFIG.data.lr_variables),
        hr_shape=tuple(CONFIG.graph.hr_shape),
    ).to(DEVICE)
    print(f"  [J8]        skip_block:    {sum(p.numel() for p in skip.parameters())} params")

# Load baseline checkpoint (will produce J18 warnings for renamed keys —
# that is the intended behavior)
print()
print("[Load] Loading baseline V5-mini checkpoint into new Path C+ architecture...")
ckpt = torch.load(_smoke_ckpt, map_location=DEVICE, weights_only=False)

# === DS Round 8 pre-condition: hard assertion on encoder_state_dict ===
import warnings as _w
_enc_sd = ckpt.get("encoder_state_dict", {})
assert len(_enc_sd) > 0, (
    f"DS audit fix: encoder_state_dict absent or empty. "
    f"Encoder would run with random init (original bug re-occurs). "
    f"Checkpoint keys: {list(ckpt.keys())[:10]}"
)
print(f"[Audit] encoder_state_dict: {len(_enc_sd)} keys found")

load_audit = {
    "encoder_state_dict_n_keys": len(_enc_sd),
    "encoder_legacy_format_detected": any(
        k.startswith("hetero_conv.") for k in _enc_sd.keys()
    ),
    "encoder_migration_applied": False,
    "skip_block_loaded_from_ckpt": False,
    "modules_loaded_strict": [],
    "modules_loaded_fallback": [],
}


def _safe_load(name_, module, sd):
    """J18 helper + DS audit capture."""
    with _w.catch_warnings(record=True) as captured:
        _w.simplefilter("always")
        try:
            module.load_state_dict(sd, strict=True)
            migration_fired = any(
                "migration" in str(w.message).lower() for w in captured
            )
            if migration_fired and name_ == "encoder":
                load_audit["encoder_migration_applied"] = True
                print(f"  [section 1.6+J3] {name_:18s} legacy state_dict migrated automatically")
            print(f"  [J18 OK] {name_:18s} loaded strict=True")
            load_audit["modules_loaded_strict"].append(name_)
            return True
        except RuntimeError as e:
            result = module.load_state_dict(sd, strict=False)
            print(f"  [J18 WARN] {name_:18s} strict=False fallback")
            print(f"             Missing: {len(result.missing_keys)} keys "
                  f"(first 3: {result.missing_keys[:3]})")
            print(f"             Unexpected: {len(result.unexpected_keys)} keys "
                  f"(first 3: {result.unexpected_keys[:3]})")
            load_audit["modules_loaded_fallback"].append(name_)
            return False


_safe_load("encoder", enc, ckpt.get("encoder_state_dict", {}))
_safe_load("rcn_cell", rcn_cell, ckpt.get("rcn_cell_state_dict", {}))
_safe_load("regression_head", rh, ckpt.get("regression_head_state_dict", {}))
_safe_load("diffusion", diff, ckpt.get("diffusion_state_dict", {}))
if skip is not None and ckpt.get("skip_block_state_dict") is not None:
    _safe_load("skip_block", skip, ckpt["skip_block_state_dict"])
    load_audit["skip_block_loaded_from_ckpt"] = True
elif skip is not None:
    print(f"  [Info] skip_block exists but NOT in ckpt (V5-mini predates J8) - random init")

A_dag = rcn_cell.A_dag.detach().cpu().clone() if hasattr(rcn_cell, "A_dag") else None
stack_v5 = {
    "encoder": enc, "rcn_runner": rcn_runner, "regression_head": rh,
    "diffusion": diff, "skip_block": skip, "A_dag": A_dag, "variant": "Oracle-smoke",
}
print()
print(f"[OK] Stack ready. A_dag shape: {A_dag.shape if A_dag is not None else None}")
print(f"     Q_phys baseline (pre-finetune): see Cell 5")


In [ ]:
# === Cell 5 : Run smoke test with full instrumentation (Batch F + F-bis + blindspots) ===
#
# IMPORTANT — Code path scoping (blindspot #1)
# This smoke exercises `scripts/finetune_stage1_bundle_b.finetune_bundle_b`
# (Bundle B fine-tune). Phase A0'' will use `src/st_cdgm/training/training_loop.
# train_epoch_stage1` from a fresh checkpoint, which has DIFFERENT gate
# management (static dag_grad_gate_value=1.0, no auto-scale ramp). A smoke
# PASS here therefore validates only the Bundle B path; A0'' is a separate
# regime that must be validated on its own JSON. See [PC7].
#
# Batch F (commit 876a082) — after smoke #3 ARTEFACT verdict :
#   1) SMOKE_EPOCHS 10 -> 15 (gate auto-scale + math prof bandwidth check)
#   2) skip_sigma_data_recalib=True -> False (AI eng FIX 3: run calibration)
#   3) Monkey-patch project_dag_floor + project_dag_spectral to track
#      pre/post snapshots per batch (Math Prof Q4)
#   4) Magnitude-aware Q_phys continuous metric (Math Prof Q1)
#   5) Per-epoch A_dag norm/asymmetry trajectory (smoke #3 smoking gun)
#
# Batch F-bis (commit 5a4b6ca) :
#   6) 3-point projection hook (pre_spectral / post_spectral / post_floor).
#   7) Smoke JSON tag exempt from PC4 (HARKing-resistant).
#
# Blindspot follow-ups (this cell + cell 6) :
#   #1   document code-path scope explicitly in the JSON
#   #2+3 replace |norm delta| > 0.05 with PHYS_MAG_GAINED (sum |A[phys,signed]|
#        net of initial), because Frobenius norm can stay flat under correct
#        mass-redistribution learning, AND lr_rcn*n_batches bandwidth caps
#        norm-delta at ~0.006 in 15 epochs (way below the 0.05 threshold).
#   #6   compute_q_phys_continuous flags collapse instead of returning 0 silent.
#   #9   skeleton F1 use the SAME adaptive threshold as compute_q_phys_adaptive.
#
import json
import warnings
from pathlib import Path
from scripts.finetune_stage1_bundle_b import finetune_bundle_b
from src.st_cdgm.training.physics_prior import (
    build_physical_mask,
    physical_prior_loss,
    VAR_LABELS,
)


# -------------------------------------------------------------------------
# Q_phys metric definitions
# -------------------------------------------------------------------------
def compute_q_phys_binary(A_dag_np, G_phys_np, threshold=0.01):
    """BINARY Q_phys (= V5-mini original metric).

    Counts sign-correct physical edges WITH magnitude > threshold.
    Entries below threshold are treated as zero (sign undefined, no match).
    Returns: (Q_phys, matches, n_phys, n_extra)
    """
    import numpy as np
    A = np.array(A_dag_np)
    np.fill_diagonal(A, 0.0)
    G = np.array(G_phys_np)
    mask = G != 0  # 5 physical edge positions
    if not mask.any():
        return 0.0, 0, 0, 0
    A_thresh = A.copy()
    A_thresh[np.abs(A_thresh) <= threshold] = 0.0
    A_sign = np.sign(A_thresh)
    G_sign = np.sign(G)
    matches = int(((A_sign == G_sign) & mask).sum())
    n_phys = int(mask.sum())
    n_extra = int(((G == 0) & (np.abs(A) > threshold) & (~np.eye(A.shape[0], dtype=bool))).sum())
    return matches / n_phys, matches, n_phys, n_extra


def compute_q_phys_adaptive(A_dag_np, G_phys_np, frac=0.3):
    """Math Prof Q1: adaptive-threshold binary Q_phys.

    threshold = max(0.01, frac * max(|A_dag|))
    DAGMA/NOTEARS convention: a "recovered" edge must have magnitude
    proportional to the largest non-zero entry. With smoke #3's A_dag.max()
    = 0.193, frac=0.3 gives threshold=0.058, which would have rejected the
    3 tiny "0.011-magnitude" edges that produced the false PASS.
    Returns (Q_phys, matches, n_phys, threshold) so the threshold is logged.
    """
    import numpy as np
    A = np.array(A_dag_np)
    np.fill_diagonal(A, 0.0)
    G = np.array(G_phys_np)
    A_max = float(np.abs(A).max())
    threshold = max(0.01, frac * A_max)
    mask = G != 0
    if not mask.any():
        return 0.0, 0, 0, threshold
    A_thresh = A.copy()
    A_thresh[np.abs(A_thresh) <= threshold] = 0.0
    A_sign = np.sign(A_thresh)
    G_sign = np.sign(G)
    matches = int(((A_sign == G_sign) & mask).sum())
    n_phys = int(mask.sum())
    return matches / n_phys, matches, n_phys, threshold


def compute_q_phys_continuous(A_dag_np, G_phys_np, *, collapse_eps=1e-12):
    """Math Prof Q4: magnitude ratio Q_phys.

    Q_phys_cont = sum(|A[phys positions, sign-correct]|) / sum(|A[off-diag]|)
    Range [0, 1]. Gaming-resistant.

    Blindspot #6: returns (Q_phys_cont, collapsed_flag). When the denominator
    is below collapse_eps, A_dag has collapsed to ~zero -- a different failure
    mode than "model legitimately failed to recover". The caller can branch on
    the flag to log/report distinctively.
    """
    import numpy as np
    A = np.array(A_dag_np)
    np.fill_diagonal(A, 0.0)
    G = np.array(G_phys_np)
    mask_phys = G != 0
    sign_correct = (np.sign(A) == np.sign(G)) & mask_phys
    num = float(np.abs(A[sign_correct]).sum())
    den = float(np.abs(A).sum())
    if den < collapse_eps:
        return 0.0, True  # collapsed flag
    return num / den, False


def compute_phys_mag_gained(A_dag_now, A_dag_init, G_phys_np):
    """Blindspot #2+3: phys-edge mass gained (signed).

    Returns sum(|A_now[phys, sign-correct]|) - sum(|A_init[phys, sign-correct]|).
    Positive when learning has REINFORCED sign-correct physical edges; this
    is what we actually care about, and it survives the Frobenius-norm
    redistribution invariance that masks training success.

    For smoke #3 (init phys mag = 5*0.007 + 2*0.18 = 0.395, final = 5*0.011
    + 2*0.19 = 0.435), this metric reports +0.04 -- a small but nonzero gain
    that the |norm delta| heuristic missed.
    """
    import numpy as np
    A_now = np.array(A_dag_now)
    A_init = np.array(A_dag_init)
    G = np.array(G_phys_np)
    np.fill_diagonal(A_now, 0.0)
    np.fill_diagonal(A_init, 0.0)
    mask_phys = G != 0
    sign_correct_now = (np.sign(A_now) == np.sign(G)) & mask_phys
    sign_correct_init = (np.sign(A_init) == np.sign(G)) & mask_phys
    return float(np.abs(A_now[sign_correct_now]).sum() -
                 np.abs(A_init[sign_correct_init]).sum())


def compute_skeleton_f1(A_dag_np, G_phys_np, threshold=None, frac=0.3):
    """Blindspot #9: skeleton F1 with adaptive threshold (default).

    Original (smoke #3) used fixed threshold=0.01 — inconsistent with the
    adaptive Q_phys. Now uses max(0.01, frac * max|A|) by default, matching
    compute_q_phys_adaptive. If `threshold` is explicitly passed, that value
    is used (legacy path for diagnostic comparison).
    """
    import numpy as np
    A = np.array(A_dag_np)
    np.fill_diagonal(A, 0.0)
    G = np.array(G_phys_np)
    if threshold is None:
        threshold = max(0.01, frac * float(np.abs(A).max()))
    A_skel = ((np.abs(A) > threshold) | (np.abs(A.T) > threshold)).astype(int)
    np.fill_diagonal(A_skel, 0)
    G_skel = ((G != 0) | (G.T != 0)).astype(int)
    np.fill_diagonal(G_skel, 0)
    iu = np.triu_indices_from(A_skel, k=1)
    A_edges = A_skel[iu]
    G_edges = G_skel[iu]
    tp = int(((A_edges == 1) & (G_edges == 1)).sum())
    fp = int(((A_edges == 1) & (G_edges == 0)).sum())
    fn = int(((A_edges == 0) & (G_edges == 1)).sum())
    if tp + fp == 0 or tp + fn == 0:
        return 0.0, threshold
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    if precision + recall == 0:
        return 0.0, threshold
    return 2 * precision * recall / (precision + recall), threshold


G_phys = build_physical_mask(num_vars=num_vars)
A_dag_initial = rcn_cell.A_dag.detach().cpu().numpy()
q_phys_initial, matches_init, n_phys, n_extra_init = compute_q_phys_binary(
    A_dag_initial, G_phys.numpy()
)
q_phys_adapt_initial, matches_init_adapt, _, thresh_adapt_initial = compute_q_phys_adaptive(
    A_dag_initial, G_phys.numpy()
)
q_phys_cont_initial, _collapse_init = compute_q_phys_continuous(A_dag_initial, G_phys.numpy())
skeleton_f1_initial, skel_thresh_initial = compute_skeleton_f1(
    A_dag_initial, G_phys.numpy()
)

print(f"[Smoke] Q_phys BINARY initial   : {q_phys_initial:.4f} ({matches_init}/{n_phys})  threshold=0.01")
print(f"[Smoke] Q_phys ADAPTIVE initial : {q_phys_adapt_initial:.4f} ({matches_init_adapt}/{n_phys})  threshold={thresh_adapt_initial:.4f}")
print(f"[Smoke] Q_phys CONTINUOUS init  : {q_phys_cont_initial:.4f}  (gaming-resistant magnitude ratio)")
print(f"[Smoke] Skeleton F1 (adaptive)  : {skeleton_f1_initial:.4f}  threshold={skel_thresh_initial:.4f}  (blindspot #9)")
print(f"[Smoke] n_extra edges (init)    : {n_extra_init}")
print(f"[Smoke] A_dag initial matrix (rounded):")
for _row in A_dag_initial:
    print(f"        {[round(float(x), 4) for x in _row]}")
print(f"[Smoke] A_dag initial norm      : {(A_dag_initial ** 2).sum() ** 0.5:.4f}")
print(f"[Smoke] A_dag initial asymmetry : {((A_dag_initial - A_dag_initial.T) ** 2).sum() ** 0.5:.4f}")
print()

# -------------------------------------------------------------------------
# Batch F-bis: 3-point projection hook (pre_spectral / post_spectral / post_floor)
# -------------------------------------------------------------------------
# Order of calls inside finetune_bundle_b (lines 482-489):
#   1) project_dag_spectral  -- caps top of the norm
#   2) project_dag_floor     -- raises the floor (optionally re-anchors prior)
#
projection_log = {
    "pre_spectral_A_dag": [],   # what the optimizer alone produced
    "post_spectral_A_dag": [],  # after project_dag_spectral
    "post_floor_A_dag": [],     # after project_dag_floor (= what model uses)
    "spectral_rescale": [],
    "floor_rescale": [],
    "batch_idx": [],
}
_orig_project_dag_spectral = rcn_cell.project_dag_spectral
_orig_project_dag_floor = rcn_cell.project_dag_floor
_batch_counter = {"n": 0}


def _patched_project_dag_spectral(max_radius=0.95):
    pre = rcn_cell.A_dag.data.detach().cpu().clone().numpy()
    rescale = _orig_project_dag_spectral(max_radius=max_radius)
    post = rcn_cell.A_dag.data.detach().cpu().clone().numpy()
    projection_log["pre_spectral_A_dag"].append(pre)
    projection_log["post_spectral_A_dag"].append(post)
    projection_log["spectral_rescale"].append(float(rescale))
    return rescale


def _patched_project_dag_floor(min_norm=0.10, prior=None):
    rescale = _orig_project_dag_floor(min_norm=min_norm, prior=prior)
    post = rcn_cell.A_dag.data.detach().cpu().clone().numpy()
    projection_log["post_floor_A_dag"].append(post)
    projection_log["floor_rescale"].append(float(rescale))
    projection_log["batch_idx"].append(_batch_counter["n"])
    _batch_counter["n"] += 1
    return rescale


rcn_cell.project_dag_spectral = _patched_project_dag_spectral
rcn_cell.project_dag_floor = _patched_project_dag_floor
print("[Batch F-bis] Projection hooks installed (3-point: pre_spectral / post_spectral / post_floor)")

epoch_A_dag = [A_dag_initial.copy()]


# -------------------------------------------------------------------------
# Smoke training config
# -------------------------------------------------------------------------
SMOKE_EPOCHS = 15
SMOKE_SANITY_EVERY = 1

_gate_start = min(5, max(2, SMOKE_EPOCHS // 8))
_gate_end = min(20, max(_gate_start + 2, SMOKE_EPOCHS // 4))

# Blindspot #2 — bandwidth math for diagnostic display
_lr_rcn = 3e-5
_force_max = 0.20  # combined L_phys + L_l1 per-entry force estimate
_batches_per_epoch_est = max(1, 100 // GPU_PROFILE.get("batch_size", 8))
_n_batches_total = _batches_per_epoch_est * SMOKE_EPOCHS
_max_per_entry_displacement = _lr_rcn * _n_batches_total * _force_max
print(f"[Smoke] Launching {SMOKE_EPOCHS} epochs Bundle B fine-tune (Batch F + F-bis)")
print(f"[Smoke] Code path: finetune_bundle_b (NOT train_epoch_stage1, see blindspot #1)")
print(f"[Smoke] Hyperparameters (Path C+ corrected):")
print(f"        lambda_l1_start  : 0.04  (was 0.10 in V5-mini)")
print(f"        lambda_l1_end    : 0.005 (was 0.01)")
print(f"        lambda_dag_prior : 0.40  (was 0.05)")
print(f"        g_phys_alpha     : 0.25  (was 0.20)")
print(f"        dag_grad_gate    : ramp 0->1 epochs {_gate_start}-{_gate_end} "
      f"(auto-scale active; gate=1.0 from epoch {_gate_end})")
print(f"        sigma_data calib : ENABLED (Batch F-2)")
print(f"[Smoke] Bandwidth estimate (blindspot #2):")
print(f"        lr_rcn={_lr_rcn}, ~{_batches_per_epoch_est} batches/epoch, {_n_batches_total} total batches")
print(f"        Max per-entry displacement = lr × batches × force ≈ {_max_per_entry_displacement:.6f}")
print(f"        Max ||A|| delta ≈ sqrt(30) × {_max_per_entry_displacement:.6f} ≈ {(30**0.5)*_max_per_entry_displacement:.4f}")
print(f"        -> The criterion '|norm delta| > 0.05' is ~{0.05/((30**0.5)*_max_per_entry_displacement):.1f}x above this bandwidth!")
print(f"        -> Smoke #4 uses PHYS_MAG_GAINED criterion instead (blindspot #3).")
print()

with warnings.catch_warnings(record=True) as w_record:
    warnings.simplefilter("always")
    result = finetune_bundle_b(
        stack=stack_v5,
        builder=builder,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        CONFIG=CONFIG,
        DEVICE=DEVICE,
        epochs=SMOKE_EPOCHS,
        batch_size=GPU_PROFILE["batch_size"],
        ckpt_save_dir=SMOKE_DIR,
        convert_sample_to_batch_fn=convert_sample_to_batch,
        sanity_eval_every=SMOKE_SANITY_EVERY,
        seed=42,
        skip_sigma_data_recalib=False,  # Batch F-2: enabled
    )

# Reconstruct per-epoch A_dag from per-batch post_floor log
if projection_log["post_floor_A_dag"]:
    n_batches_total = len(projection_log["post_floor_A_dag"])
    n_per_epoch = max(1, n_batches_total // SMOKE_EPOCHS)
    for _ep in range(SMOKE_EPOCHS):
        idx = min(n_batches_total - 1, (_ep + 1) * n_per_epoch - 1)
        epoch_A_dag.append(projection_log["post_floor_A_dag"][idx].copy())

j8_failed = [w for w in w_record if "J8 skip_block forward failed" in str(w.message)]
missing_edge = [w for w in w_record if "§1.6 missing-edge fallback" in str(w.message)]
n_spectral_fired = sum(1 for r in projection_log["spectral_rescale"] if abs(r - 1.0) > 1e-9)
n_floor_fired = sum(1 for r in projection_log["floor_rescale"] if abs(r - 1.0) > 1e-9)
print()
print(f"[Smoke] J8 reshape failures detected : {len(j8_failed)}")
print(f"[Smoke] §1.6 missing-edge fallbacks  : {len(missing_edge)}")
print(f"[Smoke] Projector SPECTRAL fired     : {n_spectral_fired} / {len(projection_log['spectral_rescale'])} batches")
print(f"[Smoke] Projector FLOOR fired        : {n_floor_fired} / {len(projection_log['floor_rescale'])} batches")
print(f"[Smoke] Per-epoch A_dag snapshots    : {len(epoch_A_dag)} (init + {SMOKE_EPOCHS} epochs)")


In [ ]:
# === Cell 6 : Verdict (Batch F + F-bis + blindspots) ===
import json
import numpy as np

# 3-point snapshots (Batch F-bis)
A_dag_final = rcn_cell.A_dag.detach().cpu().numpy()
A_dag_pre_spectral = (projection_log["pre_spectral_A_dag"][-1].copy()
                     if projection_log["pre_spectral_A_dag"] else A_dag_final.copy())
A_dag_post_spectral = (projection_log["post_spectral_A_dag"][-1].copy()
                      if projection_log["post_spectral_A_dag"] else A_dag_final.copy())
A_dag_post_floor = (projection_log["post_floor_A_dag"][-1].copy()
                   if projection_log["post_floor_A_dag"] else A_dag_final.copy())

# Q_phys variants on FINAL post-projection A_dag
q_phys_final, matches_final, n_phys, n_extra_final = compute_q_phys_binary(
    A_dag_final, G_phys.numpy()
)
q_phys_adapt_final, matches_final_adapt, _, thresh_adapt_final = compute_q_phys_adaptive(
    A_dag_final, G_phys.numpy()
)
# Blindspot #6: returns (value, collapsed)
q_phys_cont_final, collapsed_final = compute_q_phys_continuous(A_dag_final, G_phys.numpy())
skeleton_f1_final, skel_thresh_final = compute_skeleton_f1(
    A_dag_final, G_phys.numpy()
)

# Same metrics on PRE-SPECTRAL A_dag (= optimizer-only output)
q_phys_pre_spec_binary, matches_ps, _, _ = compute_q_phys_binary(
    A_dag_pre_spectral, G_phys.numpy()
)
q_phys_pre_spec_cont, collapsed_pre_spec = compute_q_phys_continuous(
    A_dag_pre_spectral, G_phys.numpy()
)

# Blindspot #2 + #3: replace PASS_NORM_MOVED with PASS_PHYS_MAG_GAINED
phys_mag_gained = compute_phys_mag_gained(A_dag_final, A_dag_initial, G_phys.numpy())
phys_mag_gained_pre_spectral = compute_phys_mag_gained(
    A_dag_pre_spectral, A_dag_initial, G_phys.numpy()
)

# Trajectory metrics (kept for diagnostic, no longer load-bearing)
asym_initial = float(((A_dag_initial - A_dag_initial.T) ** 2).sum() ** 0.5)
asym_final = float(((A_dag_final - A_dag_final.T) ** 2).sum() ** 0.5)
mag_initial = float((A_dag_initial ** 2).sum() ** 0.5)
mag_final = float((A_dag_final ** 2).sum() ** 0.5)
asym_delta = asym_final - asym_initial
mag_delta = mag_final - mag_initial

epoch_norms = [float((A ** 2).sum() ** 0.5) for A in epoch_A_dag]
epoch_asyms = [float(((A - A.T) ** 2).sum() ** 0.5) for A in epoch_A_dag]

thresh = 0.05
n_edges_initial = int((np.abs(A_dag_initial) > thresh).sum())
n_edges_final = int((np.abs(A_dag_final) > thresh).sum())

phys_mask = (G_phys.numpy() != 0)
phys_mag_initial_list = np.abs(A_dag_initial)[phys_mask].tolist()
phys_mag_final_list = np.abs(A_dag_final)[phys_mask].tolist()
spur_mag_initial = sorted(np.abs(A_dag_initial)[~phys_mask & ~np.eye(6, dtype=bool)].tolist(), reverse=True)[:7]
spur_mag_final = sorted(np.abs(A_dag_final)[~phys_mask & ~np.eye(6, dtype=bool)].tolist(), reverse=True)[:7]

# Spur-mag delta (blindspot #3: complement of phys_mag_gained)
sum_spur_initial = float(np.abs(A_dag_initial)[~phys_mask & ~np.eye(6, dtype=bool)].sum())
sum_spur_final = float(np.abs(A_dag_final)[~phys_mask & ~np.eye(6, dtype=bool)].sum())
spur_mag_delta = sum_spur_final - sum_spur_initial

print("=" * 72)
print("PATH C+ SMOKE TEST VERDICT (Batch F + F-bis + blindspots)")
print("=" * 72)
print()
print("Q_phys (4 variants):")
print(f"  BINARY (thresh=0.01)     : {q_phys_initial:.4f} ({matches_init}/{n_phys}) -> {q_phys_final:.4f} ({matches_final}/{n_phys})")
print(f"  ADAPTIVE (thresh=0.3·max): {q_phys_adapt_initial:.4f} ({matches_init_adapt}/{n_phys}) -> {q_phys_adapt_final:.4f} ({matches_final_adapt}/{n_phys})  threshold_final={thresh_adapt_final:.4f}")
print(f"  CONTINUOUS (mag ratio)   : {q_phys_cont_initial:.4f} -> {q_phys_cont_final:.4f}{' [COLLAPSED]' if collapsed_final else ''}")
print(f"  BINARY    optimizer-only : {q_phys_initial:.4f} -> {q_phys_pre_spec_binary:.4f} ({matches_ps}/{n_phys})  [pre-spectral]")
print(f"  CONTINUOUS optimizer-only: {q_phys_cont_initial:.4f} -> {q_phys_pre_spec_cont:.4f}{' [COLLAPSED]' if collapsed_pre_spec else ''}  [pre-spectral]")
print()
print("Mass redistribution (blindspot #2 + #3 — the right diagnostic):")
print(f"  Phys mag gained (signed) : {phys_mag_gained:+.4f}  [POSITIVE = sign-correct phys edges reinforced]")
print(f"  Phys mag gained (pre-sp) : {phys_mag_gained_pre_spectral:+.4f}  [pre-spectral, optimizer-only]")
print(f"  Spur mag delta           : {spur_mag_delta:+.4f}  [NEGATIVE = band-diagonal weakening]")
print()
print("Structural diagnostics:")
print(f"  n_extra edges (>0.01)  : {n_extra_init} -> {n_extra_final}")
print(f"  #edges > 0.05          : {n_edges_initial} -> {n_edges_final}")
print(f"  Skeleton F1 (adaptive) : {skeleton_f1_initial:.4f} -> {skeleton_f1_final:.4f}  threshold={skel_thresh_final:.4f}  (blindspot #9)")
print()
print("Trajectory (kept for diagnostic only — NOT a verdict input anymore):")
print(f"  A_dag norm             : {mag_initial:.4f} -> {mag_final:.4f}  (delta {mag_delta:+.4f}, was the smoke #3 verdict criterion)")
print(f"  A_dag asymmetry        : {asym_initial:.4f} -> {asym_final:.4f}  (delta {asym_delta:+.4f})")
print(f"  Per-epoch norm         : {[round(x, 3) for x in epoch_norms]}")
print(f"  Per-epoch asym         : {[round(x, 3) for x in epoch_asyms]}")
print()
print("Edge magnitudes (5 physical + top-7 spurious):")
print(f"  Phys mag initial  : {[round(x, 4) for x in phys_mag_initial_list]}")
print(f"  Phys mag final    : {[round(x, 4) for x in phys_mag_final_list]}")
print(f"  Spur mag initial  : {[round(x, 4) for x in spur_mag_initial]}")
print(f"  Spur mag final    : {[round(x, 4) for x in spur_mag_final]}")
print()
print(f"Projector activity:")
print(f"  Spectral fired    : {n_spectral_fired} / {len(projection_log['spectral_rescale'])} batches")
print(f"  Floor fired       : {n_floor_fired} / {len(projection_log['floor_rescale'])} batches  [blindspot #4: smoke #3 norm 0.53 >> floor 0.10 — projector should NOT have fired]")
print()
print("Warnings:")
print(f"  J8 reshape fails  : {len(j8_failed)}")
print(f"  §1.6 missing-edge : {len(missing_edge)}")
print()
print("A_dag FINAL (= post_floor, what model uses at inference):")
for _row in A_dag_final:
    print(f"  {[round(float(x), 4) for x in _row]}")
print()
print("A_dag PRE-SPECTRAL (= optimizer-only output, projection-free):")
for _row in A_dag_pre_spectral:
    print(f"  {[round(float(x), 4) for x in _row]}")
print()
print(f"Load audit:")
for _k, _v in load_audit.items():
    print(f"  {_k:35s} = {_v}")
print()

# -------------------------------------------------------------------------
# Blindspot #2 + #3 — REVISED PASS criteria
# -------------------------------------------------------------------------
# Replace |norm delta| > 0.05 (bandwidth-impossible in 15 epochs) with
# PHYS_MAG_GAINED > 0.005 (sign-correct phys edges should gain mass even at
# this small bandwidth, because the prior force is targeted at exactly these
# entries). Also require SPUR_MAG_DELTA <= 0 (band-diagonal should at least
# not get stronger).
#
PASS_BINARY = q_phys_final > 0.55
PASS_ADAPTIVE = q_phys_adapt_final >= 0.60
PASS_CONTINUOUS = q_phys_cont_final > 0.30   # smoke gate (PC5 A0'' gate is 0.50)
PASS_PHYS_MAG_GAINED = phys_mag_gained > 0.005   # blindspot #2+3 replaces norm_moved
PASS_SPUR_NOT_STRONGER = spur_mag_delta <= 0.001  # ε tolerance
PASS_J8 = len(j8_failed) == 0
PASS_NO_MISSING = len(missing_edge) == 0
PASS_NOT_COLLAPSED = not collapsed_final   # blindspot #6

print("=" * 72)
print("BATCH F + F-bis + BLINDSPOTS PASS CRITERIA")
print("=" * 72)
print(f"  [{('OK' if PASS_BINARY else 'KO')}] Q_phys binary > 0.55         : {q_phys_final:.4f}")
print(f"  [{('OK' if PASS_ADAPTIVE else 'KO')}] Q_phys adaptive >= 0.60      : {q_phys_adapt_final:.4f}  (thresh={thresh_adapt_final:.4f})")
print(f"  [{('OK' if PASS_CONTINUOUS else 'KO')}] Q_phys continuous > 0.30     : {q_phys_cont_final:.4f}")
print(f"  [{('OK' if PASS_PHYS_MAG_GAINED else 'KO')}] Phys mag gained > 0.005      : {phys_mag_gained:+.4f}  [blindspot #2+3]")
print(f"  [{('OK' if PASS_SPUR_NOT_STRONGER else 'KO')}] Spur mag delta <= 0.001      : {spur_mag_delta:+.4f}")
print(f"  [{('OK' if PASS_NOT_COLLAPSED else 'KO')}] A_dag not collapsed          : {not collapsed_final}  [blindspot #6]")
print(f"  [{('OK' if PASS_J8 else 'KO')}] J8 no reshape failures       : {len(j8_failed)}")
print(f"  [{('OK' if PASS_NO_MISSING else 'KO')}] No §1.6 missing-edge fallback: {len(missing_edge)}")
print()

ALL_PASS = (PASS_BINARY and PASS_ADAPTIVE and PASS_CONTINUOUS
            and PASS_PHYS_MAG_GAINED and PASS_SPUR_NOT_STRONGER
            and PASS_NOT_COLLAPSED and PASS_J8 and PASS_NO_MISSING)

if collapsed_final:
    VERDICT = "FAIL_COLLAPSE"
    print("FAIL A_dag COLLAPSED — denominator ~0. Different failure mode than 'no learning'.")
    print("   Investigate L1 weight (lambda_l1) — likely too high, killed the matrix.")
elif ALL_PASS:
    VERDICT = "PASS"
    print("OK SMOKE PASS — Path C+ fixes effective under blindspot-hardened criteria.")
    print("   GO Phase A0'' on A100 (3 seeds, fresh V5_DIR).")
    print("   NB blindspot #1: A0'' uses train_epoch_stage1 (different gate regime);")
    print("       A0'' must be re-validated on its own JSON.")
elif PASS_PHYS_MAG_GAINED and PASS_BINARY and not PASS_CONTINUOUS:
    VERDICT = "PARTIAL_BAND_DIAGONAL"
    print("WARN PARTIAL — Phys edges reinforced AND binary OK, but continuous low.")
    print("   Band-diagonal persists. Per PC6 = 'interventional sign-consistency")
    print("   achieved; sparse structural recovery not achieved'. A0'' with caveat.")
elif not PASS_PHYS_MAG_GAINED:
    VERDICT = "FAIL_NO_GAIN"
    print(f"FAIL — Phys edges did NOT gain mass (delta={phys_mag_gained:+.4f} <= 0.005).")
    print("   This means the targeted L_phys force did NOT pull phys edges up.")
    print("   Investigate: lambda_dag_prior balance, sign-flip in G_phys matrix,")
    print("   gradient flow at the RCN cell (set_dag_grad_gate path).")
else:
    VERDICT = "FAIL"
    print("FAIL — criteria mix not OK:")
    print(f"   phys_mag_gained={PASS_PHYS_MAG_GAINED}, binary={PASS_BINARY}, "
          f"adaptive={PASS_ADAPTIVE}, continuous={PASS_CONTINUOUS}, "
          f"spur_not_stronger={PASS_SPUR_NOT_STRONGER}")

# -------------------------------------------------------------------------
# Save full instrumentation to JSON
# -------------------------------------------------------------------------
smoke_results = {
    "verdict": VERDICT,
    "smoke_run": "smoke_4_batch_F_blindspots",
    "smoke_epochs": SMOKE_EPOCHS,
    "smoke_seed": 42,
    "gpu_profile": GPU_PROFILE.get("profile_id"),
    "commit_sha": (
        subprocess.check_output(shlex.split("git rev-parse HEAD")).decode().strip()
        if _IS_COLAB else None
    ),
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),

    # Blindspot #1: code-path scope explicitly noted
    "code_path_scope": {
        "training_fn": "scripts.finetune_stage1_bundle_b.finetune_bundle_b",
        "gate_mgmt": "auto-scaled ramp (schedule_lambdas)",
        "vs_a0pp_code_path": "src.st_cdgm.training.training_loop.train_epoch_stage1",
        "vs_a0pp_gate_mgmt": "static dag_grad_gate_value=1.0 (no ramp)",
        "warning": "Smoke PASS only validates Bundle B fine-tune; A0'' must "
                   "be re-validated on its own JSON with its own criteria.",
    },

    # Blindspot #2: bandwidth math recorded so reviewers see the constraint
    "bandwidth_analysis": {
        "lr_rcn": _lr_rcn,
        "estimated_batches_per_epoch": _batches_per_epoch_est,
        "n_batches_total_estimate": _n_batches_total,
        "max_per_entry_displacement": _max_per_entry_displacement,
        "max_norm_delta_estimate": float((30 ** 0.5) * _max_per_entry_displacement),
        "norm_delta_threshold_ratio": float(0.05 / ((30 ** 0.5) * _max_per_entry_displacement)),
        "note": "The retired criterion |norm delta| > 0.05 is ~8x above this "
                "bandwidth — would have caused false-FAIL even on correct learning.",
    },

    # Q_phys variants
    "q_phys_binary_initial": float(q_phys_initial),
    "q_phys_binary_final": float(q_phys_final),
    "q_phys_adaptive_initial": float(q_phys_adapt_initial),
    "q_phys_adaptive_final": float(q_phys_adapt_final),
    "q_phys_adaptive_threshold_final": float(thresh_adapt_final),
    "q_phys_continuous_initial": float(q_phys_cont_initial),
    "q_phys_continuous_final": float(q_phys_cont_final),
    "q_phys_continuous_collapsed_final": bool(collapsed_final),
    "q_phys_pre_spectral_binary": float(q_phys_pre_spec_binary),
    "q_phys_pre_spectral_continuous": float(q_phys_pre_spec_cont),
    "q_phys_pre_spectral_collapsed": bool(collapsed_pre_spec),

    # Mass redistribution (blindspot #2+3 metric)
    "phys_mag_gained": phys_mag_gained,
    "phys_mag_gained_pre_spectral": phys_mag_gained_pre_spectral,
    "spur_mag_delta": spur_mag_delta,

    # Counts
    "q_phys_matches_initial": matches_init,
    "q_phys_matches_final": matches_final,
    "q_phys_n_physical_edges": n_phys,
    "n_extra_edges_initial": n_extra_init,
    "n_extra_edges_final": n_extra_final,
    "n_edges_005_initial": n_edges_initial,
    "n_edges_005_final": n_edges_final,

    # Trajectory (kept for diagnostic; no longer load-bearing)
    "a_dag_norm_initial": mag_initial,
    "a_dag_norm_final": mag_final,
    "a_dag_norm_delta": mag_delta,
    "a_dag_asymmetry_initial": asym_initial,
    "a_dag_asymmetry_final": asym_final,
    "a_dag_asymmetry_delta": asym_delta,
    "per_epoch_norm_trajectory": epoch_norms,
    "per_epoch_asym_trajectory": epoch_asyms,

    # Edge magnitudes
    "phys_magnitudes_initial": phys_mag_initial_list,
    "phys_magnitudes_final": phys_mag_final_list,
    "spurious_top7_initial": spur_mag_initial,
    "spurious_top7_final": spur_mag_final,

    # Skeleton + warnings
    "skeleton_f1_initial": float(skeleton_f1_initial),
    "skeleton_f1_final": float(skeleton_f1_final),
    "skeleton_f1_threshold_final": float(skel_thresh_final),
    "j8_failures": len(j8_failed),
    "missing_edge_fallbacks": len(missing_edge),

    # 3-point A_dag snapshots
    "a_dag_initial_matrix": A_dag_initial.tolist(),
    "a_dag_final_matrix": A_dag_final.tolist(),
    "a_dag_pre_spectral_matrix": A_dag_pre_spectral.tolist(),
    "a_dag_post_spectral_matrix": A_dag_post_spectral.tolist(),
    "a_dag_post_floor_matrix": A_dag_post_floor.tolist(),
    "g_phys_matrix": G_phys.numpy().tolist(),

    # Projector activity (blindspot #4 cross-check)
    "projector_spectral_fired_count": int(n_spectral_fired),
    "projector_spectral_total_calls": len(projection_log["spectral_rescale"]),
    "projector_floor_fired_count": int(n_floor_fired),
    "projector_floor_total_calls": len(projection_log["floor_rescale"]),
    "projector_floor_should_fire_per_math_prof": False,
    "projector_floor_actually_fired": int(n_floor_fired) > 0,

    # Audit + criteria
    "load_audit": load_audit,
    "criteria_binary_55": PASS_BINARY,
    "criteria_adaptive_60": PASS_ADAPTIVE,
    "criteria_continuous_30": PASS_CONTINUOUS,
    "criteria_phys_mag_gained_0005": PASS_PHYS_MAG_GAINED,
    "criteria_spur_not_stronger": PASS_SPUR_NOT_STRONGER,
    "criteria_not_collapsed": PASS_NOT_COLLAPSED,
    "criteria_j8_no_fail": PASS_J8,
    "criteria_no_missing_edge": PASS_NO_MISSING,
}

# Blindspot #7 + DS amendment E: smoke JSONs MUST NOT carry Batch-D
# eligibility markers. Direct fields, no stamp_batch_d_json call.
smoke_results["schema_version"] = "path-c-plus-smoke-4-batch-F-blindspots"
smoke_results["valid_for_analysis"] = False
smoke_results["path_c_plus_batch"] = "F-blindspots-smoke"  # explicitly NOT "D"
smoke_results["smoke_exempt_from_pc4"] = True
smoke_results["j29_scheduler_type"] = str(CONFIG.diffusion.scheduler_type)
smoke_results["j29_cfg_scale"] = 1.0
# Deliberately do NOT set fixes_applied / k9_temporal_split / k5_train_window
# -- those fields are reserved for A0'' eligibility (PC4 audit gate).

_smoke_json = SMOKE_DIR / "smoke_4_batch_F_blindspots_results.json"
_smoke_json.write_text(json.dumps(smoke_results, indent=2, default=str))
print()
print(f"[Audit] Results saved to {_smoke_json}")
print(f"[Audit] schema_version = {smoke_results['schema_version']}  (PC4-incompatible by design)")
print(f"[Audit] code_path_scope.training_fn = {smoke_results['code_path_scope']['training_fn']}")
print(f"[Audit] Push to GitHub four-node-causal branch for team review")
